<a href="https://colab.research.google.com/github/AlperYildirim1/geometric-grokking/blob/main/Modular_Multiplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving


# =============================================================================
# CONFIGURATION
# =============================================================================
P = 113
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 6e-4
MAX_EPOCHS = 100_000
LOG_EVERY = 200
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TASK = 'multiplication'
SEEDS = list(range(1, 11))
OUTPUT_DIR = '/content/drive/MyDrive/multiplication_results'


# =============================================================================
# DETERMINISM
# =============================================================================
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


# =============================================================================
# DISCRETE LOGARITHM TABLE
# =============================================================================
def find_primitive_root(p):
    for g in range(2, p):
        seen = set()
        val = 1
        for _ in range(p - 1):
            val = (val * g) % p
            seen.add(val)
        if len(seen) == p - 1:
            return g
    raise ValueError(f"No primitive root found for p={p}")


def build_discrete_log_table(p):
    g = find_primitive_root(p)
    dlog = [-1] * p
    val = 1
    for k in range(p - 1):
        dlog[val] = k
        val = (val * g) % p
    return dlog, g


# =============================================================================
# DATASET
# =============================================================================
def make_dataset(p, frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)
    all_pairs = [(a, b) for a in range(p) for b in range(p)]
    rng.shuffle(all_pairs)

    n_train = int(len(all_pairs) * frac_train)
    train_x = torch.tensor([[a, b, p] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([(a * b) % p for a, b in all_pairs[:n_train]], dtype=torch.long)
    test_x = torch.tensor([[a, b, p] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([(a * b) % p for a, b in all_pairs[n_train:]], dtype=torch.long)
    return train_x, train_y, test_x, test_y


# =============================================================================
# TRANSFORMER
# =============================================================================
class Transformer(nn.Module):
    def __init__(self, p, d_model, num_heads, mlp_dim, norm_type='layernorm', normalize_hiddens=False):
        """
        norm_type: 'layernorm' | 'rmsnorm' | 'none'
        normalize_hiddens: if True, use L2 spherical normalization on residual stream
        """
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.norm_type = norm_type
        self.normalize_hiddens = normalize_hiddens

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)
        self.unembed = nn.Linear(d_model, p, bias=False)

        # Normalization layers (only allocated if needed)
        if norm_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)
        elif norm_type == 'rmsnorm':
            self.norm1 = RMSNorm(d_model)
            self.norm2 = RMSNorm(d_model)
            self.norm3 = RMSNorm(d_model)

    def apply_norm(self, x, idx):
        """Apply the appropriate normalization."""
        if self.normalize_hiddens:
            return F.normalize(x, dim=-1)
        elif self.norm_type in ('layernorm', 'rmsnorm'):
            norm = getattr(self, f'norm{idx}')
            return norm(x)
        else:
            return x

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, 1)

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_out = self.W_O(
            (F.softmax(scores, dim=-1) @ V)
            .transpose(1, 2).contiguous().view(B, L, self.d_model)
        )

        h = h + attn_out
        h = self.apply_norm(h, 2)

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, 3)

        h_final = h[:, 2, :]

        # Universal Softmax Collapse fix (Prieto et al.)
        # Normalized unembedding + fixed temperature for ALL model types
        w_normalized = F.normalize(self.unembed.weight, dim=1)
        logits = F.linear(h_final, w_normalized)
        return logits * 10.0


class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return self.scale * (x / rms)


# =============================================================================
# FOURIER DIAGNOSTICS
# =============================================================================
def extract_WL(model):
    # Always use normalized weights (matches forward pass)
    W_U_norm = F.normalize(model.unembed.weight, dim=1)
    return W_U_norm @ model.mlp_out.weight


def run_fourier_diagnostics(model, name, p, test_x, test_y, dlog):
    print(f"\n  FOURIER DIAGNOSTICS: {name}")
    model.eval()
    test_x_dev, test_y_dev = test_x.to(DEVICE), test_y.to(DEVICE)

    W_L = extract_WL(model)

    # Key frequency detection in dlog domain
    log_order = [i for i in range(1, p) if dlog[i] >= 0]
    log_order.sort(key=lambda x: dlog[x])
    W_L_reordered = W_L[log_order, :]
    fft_W_L = torch.fft.fft(W_L_reordered, dim=0)
    n_freqs = len(log_order)
    norms = torch.linalg.norm(torch.abs(fft_W_L), dim=1)[:n_freqs // 2 + 1]
    norms[0] = 0
    top_k = torch.argsort(norms, descending=True)[:5].cpu().numpy().tolist()
    print(f"    Top 5 Frequencies (dlog domain, Z_{p-1}): {top_k}")

    # Causal ablation in log-reordered space
    with torch.no_grad():
        orig_logits = model(test_x_dev)
        log_order_t = torch.tensor(log_order, device=DEVICE)
        n_log = len(log_order_t)

        logits_log = orig_logits[:, log_order_t]
        fft_logits = torch.fft.rfft(logits_log, dim=-1)

        mask = torch.zeros(fft_logits.shape[-1], dtype=torch.bool, device=DEVICE)
        mask[0] = True
        for k in top_k:
            if k < len(mask):
                mask[k] = True

        fft_restricted = fft_logits.clone()
        fft_restricted[:, ~mask] = 0
        logits_log_restricted = torch.fft.irfft(fft_restricted, n=n_log, dim=-1)

        logits_restricted = orig_logits.clone()
        for pos_idx in range(n_log):
            tok_idx = log_order[pos_idx]
            logits_restricted[:, tok_idx] = logits_log_restricted[:, pos_idx]

        restricted_acc = (logits_restricted.argmax(-1) == test_y_dev).float().mean().item()
    print(f"    Causal Ablation (Top 5 Freqs): {restricted_acc*100:.2f}%")

    # 2D Trigonometric FVE
    cache = {}
    def hook(m, i, o):
        cache['mlp'] = o.detach()
    handle = model.mlp_in.register_forward_hook(hook)

    a_vals, b_vals = torch.arange(p), torch.arange(p)
    A, B = torch.meshgrid(a_vals, b_vals, indexing='ij')
    inputs = torch.stack(
        [A.flatten(), B.flatten(), torch.full_like(A.flatten(), p)], dim=1
    ).to(DEVICE)

    with torch.no_grad():
        _ = model(inputs)
        mlp_acts = F.relu(cache['mlp'])[:, 2, :]
    handle.remove()

    dlog_tensor = torch.tensor(dlog, dtype=torch.float32)
    a_int, b_int = A.flatten(), B.flatten()
    valid_mask = (a_int > 0) & (b_int > 0)
    dlog_a = dlog_tensor[a_int[valid_mask]]
    dlog_b = dlog_tensor[b_int[valid_mask]]
    phase_sum = dlog_a + dlog_b
    mlp_acts_valid = mlp_acts[valid_mask]
    period = float(p - 1)

    n_log = len(log_order)
    c_log = torch.arange(n_log).float().to(DEVICE)

    fve_results = {}
    for k in top_k:
        wk = 2 * math.pi * k / period
        cos_c = torch.cos(2 * math.pi * k / n_log * c_log)
        sin_c = torch.sin(2 * math.pi * k / n_log * c_log)
        u_k = cos_c @ W_L_reordered
        v_k = sin_c @ W_L_reordered

        act_cos = mlp_acts_valid @ u_k
        act_sin = mlp_acts_valid @ v_k

        ideal_cos = torch.cos(wk * phase_sum).to(DEVICE)
        ideal_sin = torch.sin(wk * phase_sum).to(DEVICE)

        def calc_fve_2d(actual, ic, isn):
            y = actual - actual.mean()
            x1 = ic - ic.mean()
            x2 = isn - isn.mean()
            X = torch.stack([x1, x2], dim=1)
            beta = torch.linalg.lstsq(X, y).solution
            fve = 1.0 - (torch.var(y - (X @ beta)) / torch.var(y))
            return max(0.0, fve.item() * 100.0)

        fve_u = calc_fve_2d(act_cos, ideal_cos, ideal_sin)
        fve_v = calc_fve_2d(act_sin, ideal_cos, ideal_sin)
        fve_results[k] = (fve_u, fve_v)
        print(f"    Freq {k:<4} | U: {fve_u:>6.2f}%  V: {fve_v:>6.2f}%")

    return {
        'top_k': top_k,
        'ablation_acc': restricted_acc,
        'fve': fve_results,
    }


# =============================================================================
# TRAINING
# =============================================================================
def train_model(model, name, train_x, train_y, test_x, test_y, weight_decay, seed, save_dir):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=weight_decay, betas=(0.9, 0.999)
    )
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    epoch_log = []
    grok_95 = None
    grok_100 = None

    pbar = tqdm(range(MAX_EPOCHS), desc=f"  [{name} | seed={seed}]", leave=True)

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == MAX_EPOCHS - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()
                test_loss = criterion(test_logits, test_y).item()
                train_loss = loss.item()

            epoch_log.append({
                'epoch': epoch,
                'train_acc': train_acc,
                'test_acc': test_acc,
                'train_loss': train_loss,
                'test_loss': test_loss,
            })

            pbar.set_postfix({
                'tr': f"{train_acc:.3f}",
                'te': f"{test_acc:.3f}",
                'loss': f"{train_loss:.4f}"
            })

            # Log 95% milestone
            if grok_95 is None and test_acc >= 0.95:
                grok_95 = epoch
                tqdm.write(f"    ✦ {name} seed={seed}: 95% generalization at epoch {epoch}")

            # Interrupt at 100% generalization
            if grok_100 is None and test_acc >= 1.0 - 1e-6:
                grok_100 = epoch
                tqdm.write(f"    ⚡ {name} seed={seed}: 100% generalization at epoch {epoch} — stopping.")
                break

    # --- Generate individual training curve chart ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    epochs_arr = [e['epoch'] for e in epoch_log]
    train_accs = [e['train_acc'] for e in epoch_log]
    test_accs = [e['test_acc'] for e in epoch_log]
    train_losses = [e['train_loss'] for e in epoch_log]
    test_losses = [e['test_loss'] for e in epoch_log]

    # Accuracy panel
    ax1.plot(epochs_arr, train_accs, label='Train Acc', color='#3498db', linewidth=1.5)
    ax1.plot(epochs_arr, test_accs, label='Test Acc', color='#e74c3c', linewidth=1.5)
    ax1.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')
    ax1.axhline(y=1.0, color='green', linestyle='--', alpha=0.3)
    if grok_95 is not None:
        ax1.axvline(x=grok_95, color='orange', linestyle=':', alpha=0.7, label=f'95% @ {grok_95}')
    if grok_100 is not None:
        ax1.axvline(x=grok_100, color='green', linestyle=':', alpha=0.7, label=f'100% @ {grok_100}')
    ax1.set_ylabel("Accuracy", fontsize=12)
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(fontsize=9, loc='lower right')
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f"{name} — Seed {seed} — Modular Multiplication (Z₁₁₃)", fontsize=13)

    # Loss panel
    ax2.plot(epochs_arr, train_losses, label='Train Loss', color='#3498db', linewidth=1.5)
    ax2.plot(epochs_arr, test_losses, label='Test Loss', color='#e74c3c', linewidth=1.5)
    ax2.set_ylabel("Loss", fontsize=12)
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_yscale('log')
    ax2.legend(fontsize=9, loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '').replace('.', '')
    chart_path = os.path.join(save_dir, f"chart_{safe_name}_seed{seed}.png")
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'name': name,
        'seed': seed,
        'grok_95': grok_95,
        'grok_100': grok_100,
        'final_test_acc': epoch_log[-1]['test_acc'] if epoch_log else 0.0,
        'peak_test_acc': max(e['test_acc'] for e in epoch_log) if epoch_log else 0.0,
        'chart_path': chart_path,
        'log': epoch_log,
    }


# =============================================================================
# EXPERIMENT CONFIGS
# =============================================================================
CONFIGS = [
    {
        "name": "LayerNorm Baseline",
        "norm_type": "layernorm",
        "normalize_hiddens": False,
        "weight_decay": 1.0,
    },
    {
        "name": "RMSNorm Baseline",
        "norm_type": "rmsnorm",
        "normalize_hiddens": False,
        "weight_decay": 1.0,
    },
    {
        "name": "Bounded Sphere (WD=1.0)",
        "norm_type": "none",
        "normalize_hiddens": True,
        "weight_decay": 1.0,
    },
    {
        "name": "Bounded Sphere (WD=0.0)",
        "norm_type": "none",
        "normalize_hiddens": True,
        "weight_decay": 0.0,
    },
]


# =============================================================================
# SUMMARY TABLE
# =============================================================================
def print_summary_table(all_results):
    print(f"\n{'='*100}")
    print(f"  FINAL SUMMARY — MODULAR MULTIPLICATION (Z_{P}) — {len(SEEDS)} SEEDS")
    print(f"{'='*100}")

    # Group by config
    by_config = {}
    for r in all_results:
        by_config.setdefault(r['name'], []).append(r)

    header = f"{'Architecture':<30} | {'Mean Grok':<12} | {'Std Dev':<10} | {'Min':<8} | {'Max':<8} | {'Failures':<10} | {'Peak Acc':<10}"
    print(header)
    print("-" * len(header))

    summary_data = []

    for cfg in CONFIGS:
        name = cfg['name']
        runs = by_config.get(name, [])

        # Use grok_100 for the table (100% generalization)
        successful = [r for r in runs if r['grok_100'] is not None]
        failed = [r for r in runs if r['grok_100'] is None]

        peak_accs = [r['peak_test_acc'] for r in runs]
        mean_peak = np.mean(peak_accs) * 100 if peak_accs else 0

        if successful:
            grok_epochs = [r['grok_100'] for r in successful]
            mean_g = np.mean(grok_epochs)
            std_g = np.std(grok_epochs)
            min_g = np.min(grok_epochs)
            max_g = np.max(grok_epochs)
            print(f"{name:<30} | {mean_g:<12.0f} | {std_g:<10.0f} | {min_g:<8} | {max_g:<8} | {len(failed):<2} / {len(runs):<5} | {mean_peak:.2f}%")
        else:
            print(f"{name:<30} | {'Failed':<12} | {'—':<10} | {'—':<8} | {'—':<8} | {len(failed):<2} / {len(runs):<5} | {mean_peak:.2f}%")

        summary_data.append({
            'name': name,
            'n_success': len(successful),
            'n_fail': len(failed),
            'grok_epochs': [r['grok_100'] for r in successful],
            'grok_95_epochs': [r['grok_95'] for r in runs if r['grok_95'] is not None],
            'peak_accs': peak_accs,
        })

    # Also print 95% table
    print(f"\n{'='*100}")
    print(f"  95% GENERALIZATION MILESTONES")
    print(f"{'='*100}")
    header2 = f"{'Architecture':<30} | {'Mean 95%':<12} | {'Std Dev':<10} | {'Min':<8} | {'Max':<8} | {'Hits':<10}"
    print(header2)
    print("-" * len(header2))

    for sd in summary_data:
        g95 = sd['grok_95_epochs']
        if g95:
            print(f"{sd['name']:<30} | {np.mean(g95):<12.0f} | {np.std(g95):<10.0f} | {np.min(g95):<8} | {np.max(g95):<8} | {len(g95):<2} / {len(SEEDS)}")
        else:
            print(f"{sd['name']:<30} | {'—':<12} | {'—':<10} | {'—':<8} | {'—':<8} | 0  / {len(SEEDS)}")

    # Per-seed breakdown
    print(f"\n{'='*100}")
    print(f"  PER-SEED BREAKDOWN (100% generalization epoch)")
    print(f"{'='*100}")
    header3 = f"{'Architecture':<30}" + "".join(f" | {'S'+str(s):<7}" for s in SEEDS)
    print(header3)
    print("-" * len(header3))

    for cfg in CONFIGS:
        name = cfg['name']
        runs = by_config.get(name, [])
        run_by_seed = {r['seed']: r for r in runs}
        row = f"{name:<30}"
        for s in SEEDS:
            r = run_by_seed.get(s)
            if r and r['grok_100'] is not None:
                row += f" | {r['grok_100']:<7}"
            else:
                row += f" | {'FAIL':<7}"
        print(row)

    return summary_data


# =============================================================================
# COMBINED OVERLAY CHART
# =============================================================================
def plot_combined_chart(all_results, save_dir):
    """One overlay chart per config showing all seeds."""
    by_config = {}
    for r in all_results:
        by_config.setdefault(r['name'], []).append(r)

    for cfg_name, runs in by_config.items():
        fig, ax = plt.subplots(figsize=(14, 6))
        cmap = plt.cm.tab10

        for i, r in enumerate(sorted(runs, key=lambda x: x['seed'])):
            epochs_arr = [e['epoch'] for e in r['log']]
            test_accs = [e['test_acc'] for e in r['log']]
            ax.plot(epochs_arr, test_accs, color=cmap(i), linewidth=1.2,
                    alpha=0.8, label=f"Seed {r['seed']}")

        ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
        ax.axhline(y=1.0, color='green', linestyle='--', alpha=0.3)
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel("Test Accuracy", fontsize=12)
        ax.set_title(f"{cfg_name} — All Seeds — Modular Multiplication (Z₁₁₃)", fontsize=13)
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8, ncol=5, loc='lower right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        safe_name = cfg_name.replace(' ', '_').replace('(', '').replace(')', '').replace('=', '').replace('.', '')
        path = os.path.join(save_dir, f"overlay_{safe_name}_all_seeds.png")
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Saved overlay chart: {path}")

    # Grand comparison: one line per config (mean + std band)
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['#e74c3c', '#9b59b6', '#3498db', '#2ecc71']

    for idx, cfg in enumerate(CONFIGS):
        runs = by_config.get(cfg['name'], [])
        if not runs:
            continue

        # Align to common epoch grid
        max_ep = max(max(e['epoch'] for e in r['log']) for r in runs)
        epoch_grid = list(range(0, max_ep + 1, LOG_EVERY))

        all_accs = []
        for r in runs:
            acc_dict = {e['epoch']: e['test_acc'] for e in r['log']}
            accs = []
            last_acc = 0.0
            for ep in epoch_grid:
                if ep in acc_dict:
                    last_acc = acc_dict[ep]
                accs.append(last_acc)
            all_accs.append(accs)

        # Truncate to shortest
        min_len = min(len(a) for a in all_accs)
        all_accs = [a[:min_len] for a in all_accs]
        epoch_grid = epoch_grid[:min_len]

        arr = np.array(all_accs)
        mean_acc = arr.mean(axis=0)
        std_acc = arr.std(axis=0)

        c = colors[idx % len(colors)]
        ax.plot(epoch_grid, mean_acc, color=c, linewidth=2, label=cfg['name'])
        ax.fill_between(epoch_grid, mean_acc - std_acc, mean_acc + std_acc,
                        color=c, alpha=0.15)

    ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Test Accuracy (mean ± std)", fontsize=12)
    ax.set_title(f"All Architectures — Modular Multiplication (Z₁₁₃) — {len(SEEDS)} Seeds", fontsize=13)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    path = os.path.join(save_dir, "grand_comparison_all_configs.png")
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved grand comparison: {path}")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"{'='*70}")
    print(f"  MODULAR MULTIPLICATION EXPERIMENT (Z_{P})")
    print(f"  Seeds: {SEEDS}")
    print(f"  Configs: {len(CONFIGS)}")
    print(f"  Max epochs: {MAX_EPOCHS:,}")
    print(f"  LR: {LR} | Device: {DEVICE}")
    print(f"  Output: {OUTPUT_DIR}/")
    print(f"{'='*70}")

    dlog, prim_root = build_discrete_log_table(P)
    print(f"  Primitive root of {P}: g = {prim_root}")

    all_results = []
    total_runs = len(CONFIGS) * len(SEEDS)
    run_count = 0

    for cfg in CONFIGS:
        for seed in SEEDS:
            run_count += 1
            print(f"\n{'━'*70}")
            print(f"  RUN {run_count}/{total_runs}: {cfg['name']} | Seed {seed}")
            print(f"{'━'*70}")

            # Dataset — same split per seed
            tr_x, tr_y, te_x, te_y = make_dataset(P, FRAC_TRAIN, seed=seed)

            # Model
            set_seed(seed)
            model = Transformer(
                P, D_MODEL, NUM_HEADS, MLP_DIM,
                norm_type=cfg['norm_type'],
                normalize_hiddens=cfg['normalize_hiddens'],
            ).to(DEVICE)

            # Train
            result = train_model(
                model, cfg['name'], tr_x, tr_y, te_x, te_y,
                weight_decay=cfg['weight_decay'],
                seed=seed,
                save_dir=OUTPUT_DIR,
            )

            # Fourier diagnostics (only if model generalized)
            if result['grok_100'] is not None or result['peak_test_acc'] > 0.90:
                diag = run_fourier_diagnostics(model, cfg['name'], P, te_x, te_y, dlog)
                result['diagnostics'] = diag
            else:
                print(f"    Skipping diagnostics (peak acc: {result['peak_test_acc']*100:.1f}%)")

            all_results.append(result)

            # Save incremental results
            save_data = []
            for r in all_results:
                save_r = {k: v for k, v in r.items() if k != 'log'}
                if 'diagnostics' in save_r:
                    # Convert numpy/tensor to python types
                    d = save_r['diagnostics']
                    save_r['diagnostics'] = {
                        'top_k': d['top_k'],
                        'ablation_acc': float(d['ablation_acc']),
                    }
                save_data.append(save_r)

            with open(os.path.join(OUTPUT_DIR, "results_incremental.json"), 'w') as f:
                json.dump(save_data, f, indent=2, default=str)

    # Final summary
    summary = print_summary_table(all_results)

    # Combined charts
    print(f"\n  Generating combined charts...")
    plot_combined_chart(all_results, OUTPUT_DIR)

    # Save final summary
    with open(os.path.join(OUTPUT_DIR, "summary.txt"), 'w') as f:
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            print_summary_table(all_results)
        f.write(buf.getvalue())

    print(f"\n{'='*70}")
    print(f"  ALL DONE. Results saved to {OUTPUT_DIR}/")
    print(f"{'='*70}")